# Time To Entry Repeated Split Experiments

Ce notebook reprend le script `time_to_entry_repeated_split_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Repete l'experience time-to-entry sur splits multiples pour robustesse live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Repeated-split time-to-entry experiments on the same seeds used by the old causal audit.
- Run par defaut : `runs/exp_102_time_to_entry_repeated_split_compare`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "time_to_entry_repeated_split_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from ml_pipeline import ROOT, RUNS_DIR, safe_auc, write_json
from time_to_entry_experiments import (
    SURVIVAL_HORIZONS,
    add_selection_score,
    ap_auc_rows,
    create_run_dir,
    device_from_arg,
    evaluate_causal,
    future_entry_target,
    load_sequence_data,
    make_multibin_targets,
    make_survival_targets,
    multibin_score_frame,
    score_columns_for_objective,
    survival_score_frame,
    train_one,
)
from sequence_experiments import set_seed


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `normalize_for_split`

Cette cellule definit `normalize_for_split`. Elle prepare une partie du script.

In [ ]:
def normalize_for_split(X_raw, meta):
    train_mask = meta["split"].to_numpy() == "train"
    flat = X_raw[train_mask].reshape(-1, X_raw.shape[-1])
    mean = flat.mean(axis=0)
    std = flat.std(axis=0)
    std = np.where(std > 1e-6, std, 1.0)
    return ((X_raw - mean) / std).astype(np.float32), mean.astype(np.float32), std.astype(np.float32)


## Fonction `load_raw_sequence_data`

Cette cellule definit `load_raw_sequence_data`. Elle prepare une partie du script.

In [ ]:
def load_raw_sequence_data(sequence_run):
    sequence_run = resolve(sequence_run)
    data = np.load(sequence_run / "features" / "sequence_dataset.npz")
    X_norm = data["X"].astype(np.float32)
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    return X_norm * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)


## Fonction `repeated_split_maps`

Cette cellule definit `repeated_split_maps`. Elle prepare une partie du script.

In [ ]:
def repeated_split_maps(score_run):
    score_run = resolve(score_run)
    maps = {}
    for path in sorted((score_run / "features").glob("final_scores_seed*.csv")):
        seed = int(path.stem.replace("final_scores_seed", ""))
        df = pd.read_csv(path, usecols=["video_id", "split"]).drop_duplicates()
        maps[seed] = dict(zip(df["video_id"], df["split"]))
    if not maps:
        raise SystemExit(f"No final_scores_seed*.csv files found in {score_run}")
    return maps


## Fonction `evaluate_prediction_frame`

Cette cellule definit `evaluate_prediction_frame`. Elle prepare une partie du script.

In [ ]:
def evaluate_prediction_frame(pred, score_cols, thresholds, persistence_values, repeat_seed):
    rows = []
    for score_col in score_cols:
        for threshold in thresholds:
            for persistence in persistence_values:
                for split in ["val", "test"]:
                    row = evaluate_causal(pred, score_col, threshold, split, persistence)
                    row["objective"] = str(pred["objective"].iloc[0])
                    row["model_name"] = str(pred["model_name"].iloc[0])
                    row["inference_ms_per_window"] = float(pred["inference_ms_per_window"].iloc[0])
                    row["train_time_s"] = float(pred["train_time_s"].iloc[0])
                    row["repeat_seed"] = int(repeat_seed)
                    rows.append(row)
    return rows


## Fonction `evaluate_old_baseline`

Cette cellule definit `evaluate_old_baseline`. Elle prepare une partie du script.

In [ ]:
def evaluate_old_baseline(score_run, thresholds, persistence_values):
    score_run = resolve(score_run)
    rows = []
    for path in sorted((score_run / "features").glob("final_scores_seed*.csv")):
        seed = int(path.stem.replace("final_scores_seed", ""))
        pred = pd.read_csv(path)
        pred["objective"] = "old_final_score"
        pred["model_name"] = "old_final_sequence_only"
        pred["train_time_s"] = 0.0
        pred["inference_ms_per_window"] = 0.0
        rows.extend(evaluate_prediction_frame(pred, ["final_sequence_only"], thresholds, persistence_values, seed))
    return rows


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(df, group_cols, metric_cols):
    rows = []
    for keys, group in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        row["danger_total"] = int(group["danger_videos"].sum()) if "danger_videos" in group else 0
        row["pre_entry_detected_total"] = int(group["pre_entry_detected"].sum()) if "pre_entry_detected" in group else 0
        row["early_03_detected_total"] = int(group["early_03_detected"].sum()) if "early_03_detected" in group else 0
        row["early_05_detected_total"] = int(group["early_05_detected"].sum()) if "early_05_detected" in group else 0
        row["false_alarm_episodes_total"] = int(group["false_alarm_episodes"].sum()) if "false_alarm_episodes" in group else 0
        for col in metric_cols:
            vals = pd.to_numeric(group[col], errors="coerce")
            row[f"{col}_mean"] = float(vals.mean())
            row[f"{col}_std"] = float(vals.std(ddof=0))
        rows.append(row)
    return pd.DataFrame(rows)


## Fonction `with_base_model_name`

Cette cellule definit `with_base_model_name`. Elle prepare une partie du script.

In [ ]:
def with_base_model_name(df):
    out = df.copy()
    out["base_model_name"] = out["model_name"].astype(str).str.replace(r"^seed\d+_", "", regex=True)
    return out


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, selected_test, best_new_test, best_old_test):
    lines = [
        "# Repeated-Split Time-to-Entry Comparison",
        "",
        "This run trains the new multi-bin and survival-style models on the same repeated parent-video test splits used by the old `17/26` causal half-second audit.",
        "",
        "## Validation-Selected New Policy Applied To Repeated Test Splits",
        "",
        "| field | value |",
        "|---|---:|",
    ]
    for key in [
        "objective",
        "base_model_name",
        "score_col",
        "threshold",
        "persistence_windows",
        "selection_score_mean",
        "pre_entry_recall_mean",
        "early_03_recall_mean",
        "early_05_recall_mean",
        "event_precision_mean",
        "false_alarms_per_min_mean",
    ]:
        value = selected_test.get(key, "")
        if isinstance(value, float):
            value = f"{value:.3f}"
        lines.append(f"| {key} | {value} |")

    lines.extend([
        "",
        "## Best Repeated-Split Test Rows",
        "",
        "| family | model | score | threshold | persist | >=0.3s | >=0.5s | early/danger | precision | FA/min |",
        "|---|---|---|---:|---:|---:|---:|---:|---:|---:|",
    ])
    for _, row in best_new_test.head(12).iterrows():
        lines.append(
            f"| new | {row['base_model_name']} | {row['score_col']} | {row['threshold']:.2f} | {int(row['persistence_windows'])} | "
            f"{row['early_03_recall_mean']:.3f} | {row['early_05_recall_mean']:.3f} | "
            f"{int(row['early_05_detected_total'])}/{int(row['danger_total'])} | {row['event_precision_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} |"
        )
    for _, row in best_old_test.head(6).iterrows():
        lines.append(
            f"| old | {row['base_model_name']} | {row['score_col']} | {row['threshold']:.2f} | {int(row['persistence_windows'])} | "
            f"{row['early_03_recall_mean']:.3f} | {row['early_05_recall_mean']:.3f} | "
            f"{int(row['early_05_detected_total'])}/{int(row['danger_total'])} | {row['event_precision_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} |"
        )

    lines.extend([
        "",
        "## Notes",
        "",
        "- Denominator is repeated-split test danger clips across seeds 111/222/333, matching the old causal audit protocol.",
        "- This still operates on the 69 represented videos used by the old final-score run; 3 annotated danger clips never had `window_features` upstream and are absent from both protocols.",
        "",
        "## Artifacts",
        "",
        f"- Summary metrics: `{run_dir / 'metrics' / 'repeated_time_to_entry_summary.csv'}`",
        f"- Raw causal metrics: `{run_dir / 'metrics' / 'repeated_time_to_entry_causal_metrics.csv'}`",
        f"- Training summary: `{run_dir / 'metrics' / 'repeated_time_to_entry_training_summary.csv'}`",
    ])
    (run_dir / "repeated_time_to_entry_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    set_seed(args.seed)
    device = device_from_arg(args.device)
    run_dir = create_run_dir(args.run_name)
    X_raw = load_raw_sequence_data(args.sequence_run)
    _, base_meta, sequence_run, base_run = load_sequence_data(args.sequence_run, args.base_run)
    split_maps = repeated_split_maps(args.old_score_run)
    thresholds = [round(float(x), 2) for x in np.arange(args.threshold_min, args.threshold_max + 1e-9, args.threshold_step)]
    persistence_values = [int(x) for x in args.persistence_windows]

    multibin_y, multibin_weights = make_multibin_targets(base_meta)
    survival_y, survival_weights = make_survival_targets(base_meta)

    write_json(
        run_dir / "config.json",
        {
            "sequence_run": str(sequence_run),
            "base_run": str(base_run),
            "old_score_run": str(resolve(args.old_score_run)),
            "device": str(device),
            "seeds": sorted(split_maps),
            "model_kinds": args.model_kinds,
            "thresholds": thresholds,
            "persistence_values": persistence_values,
        },
    )

    specs = []
    for kind in args.model_kinds:
        specs.append(("multibin", kind, f"multibin_{kind}_aug"))
    for kind in args.model_kinds:
        specs.append(("survival", kind, f"survival_{kind}_aug"))

    all_metric_rows = []
    all_ap_rows = []
    all_history = []
    train_rows = []
    split_audit_rows = []

    for repeat_seed, split_map in sorted(split_maps.items()):
        meta = base_meta.copy()
        meta["split"] = meta["video_id"].map(split_map)
        if meta["split"].isna().any():
            missing = sorted(meta.loc[meta["split"].isna(), "video_id"].unique().tolist())
            raise RuntimeError(f"missing split assignments for seed {repeat_seed}: {missing[:5]}")
        X, mean, std = normalize_for_split(X_raw, meta)
        meta.to_csv(run_dir / "features" / f"split_seed_{repeat_seed}.csv", index=False)
        np.savez_compressed(run_dir / "features" / f"normalizer_seed_{repeat_seed}.npz", mean=mean, std=std)
        split_audit_rows.append(
            {
                "repeat_seed": int(repeat_seed),
                **{f"{split}_videos": int(meta[meta['split'].eq(split)]['video_id'].nunique()) for split in ['train', 'val', 'test']},
                **{f"{split}_danger_videos": int(meta[(meta['split'].eq(split)) & (meta['is_danger_clip'].eq(1))]['video_id'].nunique()) for split in ['train', 'val', 'test']},
            }
        )
        for objective, kind, name in specs:
            y = multibin_y if objective == "multibin" else survival_y
            sample_weights = multibin_weights if objective == "multibin" else survival_weights
            run_name = f"seed{repeat_seed}_{name}"
            print(f"training {run_name} on {device}")
            model, history, train_time_s, model_size_bytes, best = train_one(
                objective, kind, run_name, X, y, sample_weights, meta, run_dir, args, device
            )
            for row in history:
                row["repeat_seed"] = int(repeat_seed)
            all_history.extend(history)

            start = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() and str(device).startswith("cuda") else None
            end = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() and str(device).startswith("cuda") else None
            if start is not None:
                start.record()
            logits = []
            model.eval()
            with torch.no_grad():
                for idx in range(0, len(X), args.batch_size):
                    xb = torch.from_numpy(X[idx : idx + args.batch_size]).to(device)
                    logits.append(model(xb).detach().cpu())
            if end is not None:
                end.record()
                torch.cuda.synchronize()
                inference_s = float(start.elapsed_time(end) / 1000.0)
            else:
                inference_s = 0.0
            logits = torch.cat(logits, dim=0).numpy()

            if objective == "multibin":
                probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
                pred = multibin_score_frame(meta, probs, run_name, train_time_s, inference_s)
            else:
                probs = 1.0 / (1.0 + np.exp(-logits))
                pred = survival_score_frame(meta, probs, run_name, train_time_s, inference_s)

            pred["repeat_seed"] = int(repeat_seed)
            pred.to_csv(run_dir / "features" / f"predictions_{run_name}.csv", index=False)
            score_cols = score_columns_for_objective(objective)
            all_metric_rows.extend(evaluate_prediction_frame(pred, score_cols, thresholds, persistence_values, repeat_seed))
            for row in ap_auc_rows(pred, score_cols):
                row["repeat_seed"] = int(repeat_seed)
                all_ap_rows.append(row)
            train_rows.append(
                {
                    "repeat_seed": int(repeat_seed),
                    "objective": objective,
                    "model_name": run_name,
                    "kind": kind,
                    "best_epoch": int(best["epoch"]),
                    "best_val_ap_future_0_15s": float(best["score"]),
                    "train_time_s": float(train_time_s),
                    "model_size_bytes": int(model_size_bytes),
                    "inference_ms_per_window": float(inference_s * 1000.0 / max(len(meta), 1)),
                }
            )

    old_rows = evaluate_old_baseline(args.old_score_run, thresholds, persistence_values)
    all_metric_rows.extend(old_rows)

    metrics = add_selection_score(pd.DataFrame(all_metric_rows))
    ap_metrics = pd.DataFrame(all_ap_rows)
    history = pd.DataFrame(all_history)
    training = pd.DataFrame(train_rows)
    split_audit = pd.DataFrame(split_audit_rows)

    metrics.to_csv(run_dir / "metrics" / "repeated_time_to_entry_causal_metrics.csv", index=False)
    ap_metrics.to_csv(run_dir / "metrics" / "repeated_time_to_entry_ap_metrics.csv", index=False)
    history.to_csv(run_dir / "metrics" / "repeated_time_to_entry_training_history.csv", index=False)
    training.to_csv(run_dir / "metrics" / "repeated_time_to_entry_training_summary.csv", index=False)
    split_audit.to_csv(run_dir / "metrics" / "repeated_time_to_entry_split_audit.csv", index=False)

    metrics = with_base_model_name(metrics)
    summary = summarize(
        metrics,
        ["objective", "base_model_name", "score_col", "threshold", "persistence_windows", "split"],
        ["selection_score", "pre_entry_recall", "early_03_recall", "early_05_recall", "event_precision", "false_alarms_per_min", "median_early_warning_s"],
    )
    summary.to_csv(run_dir / "metrics" / "repeated_time_to_entry_summary.csv", index=False)

    new_test = summary[(summary["objective"] != "old_final_score") & (summary["split"] == "test")].copy()
    old_test = summary[(summary["objective"] == "old_final_score") & (summary["split"] == "test")].copy()
    val_new = summary[(summary["objective"] != "old_final_score") & (summary["split"] == "val")].copy()
    selected_val = val_new.sort_values("selection_score_mean", ascending=False).iloc[0]
    selected_test = new_test[
        new_test["objective"].eq(selected_val["objective"])
        & new_test["base_model_name"].eq(selected_val["base_model_name"])
        & new_test["score_col"].eq(selected_val["score_col"])
        & new_test["threshold"].eq(selected_val["threshold"])
        & new_test["persistence_windows"].eq(selected_val["persistence_windows"])
    ].iloc[0].to_dict()
    best_new_test = new_test.sort_values("selection_score_mean", ascending=False)
    best_old_test = old_test.sort_values("selection_score_mean", ascending=False)

    write_summary(run_dir, selected_test, best_new_test, best_old_test)
    print(run_dir)
    print(run_dir / "repeated_time_to_entry_summary.md")


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Repeated-split time-to-entry experiments on the same seeds used by the old causal audit.")
    parser.add_argument("--sequence-run", default="runs/exp_072_physical_entry_seq30_focal")
    parser.add_argument("--base-run", default="runs/exp_070_physical_entry_baseline")
    parser.add_argument("--old-score-run", default="runs/exp_088_physical_entry_final_score_full")
    parser.add_argument("--run-name", default="exp_102_time_to_entry_repeated_split_compare")
    parser.add_argument("--model-kinds", nargs="+", default=["tcn", "cnn1d", "gru", "cnn_gru"])
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--patience", type=int, default=6)
    parser.add_argument("--batch-size", type=int, default=128)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--label-smoothing", type=float, default=0.04)
    parser.add_argument("--focal-gamma", type=float, default=1.5)
    parser.add_argument("--threshold-min", type=float, default=0.05)
    parser.add_argument("--threshold-max", type=float, default=0.95)
    parser.add_argument("--threshold-step", type=float, default=0.05)
    parser.add_argument("--persistence-windows", nargs="+", type=int, default=[1, 2])
    parser.add_argument("--device", default="auto")
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_102_time_to_entry_repeated_split_compare_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["time_to_entry_repeated_split_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
